In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Column
from pyspark.sql.window import Window
from delta.tables import DeltaTable
from datetime import datetime as datum_vreme


## OpenAQ Locations

In [0]:
bronze_openaq_locations = spark.readStream.table("bg_traffic.bg_traffic_bronze.openaq_locations")


In [0]:
locations_checkpoint = "/Volumes/bg_traffic/bg_traffic_silver/checkpoints/openaq_locations"

### Flattening

In [0]:
locations_flatten = bronze_openaq_locations\
    .select(
        F.col("fetched_at"),
        F.col("_ingestion_timestamp"),
        F.col("_source_file"),
        F.explode("data.results").alias("location")
    ).select(
        "fetched_at","_ingestion_timestamp","_source_file",
        F.col("location.id").alias("location_id"),
        F.col("location.name").alias("location_name"),
        F.col("location.coordinates.latitude").alias("latitude"),
        F.col("location.coordinates.longitude").alias("longitude"),
        F.col("location.provider.id").alias("provider_id"),
        F.col("location.provider.name").alias("provider_name")

    )


In [0]:
locations_flatten.printSchema()

### Casting

In [0]:
locations_types = locations_flatten\
    .withColumn("fetched_at",F.to_timestamp("fetched_at"))\
    .withColumn("location_id", F.col("location_id").cast("long"))\
    .withColumn("provider_id", F.col("provider_id").cast("long"))\
    .withColumn("latitude", F.col("latitude").cast("double"))\
    .withColumn("longitude", F.col("longitude").cast("double"))

locations_types.printSchema()


### Validate

In [0]:
locations_valid = locations_types.filter(
    (F.col("location_id").isNotNull()) &
    (F.col("location_name").isNotNull()) &
    (F.col("latitude").between(-90,90)) &
    (F.col("longitude").between(-180,180))
)

### Merge & Dedup

In [0]:
def merge_locations(df_source, batch_id):
    if df_source.isEmpty():
        return
    
    locations_window = Window.partitionBy("location_id").orderBy(F.col("fetched_at").desc())

    locations_dedup = df_source\
    .withColumn("row_num",F.row_number().over(locations_window))\
        .filter(F.col("row_num") == 1)\
            .drop("row_num")

    silver_table_locations = DeltaTable.forName(
        spark,
        "bg_traffic.bg_traffic_silver.openaq_locations"
    ).alias("target").merge(locations_dedup.alias("source"), "target.location_id = source.location_id")\
        .whenMatchedUpdateAll()\
            .whenNotMatchedInsertAll()\
                .execute()


In [0]:
query_locations = (
    locations_valid
    .writeStream
    .foreachBatch(merge_locations)
    .option("checkpointLocation", locations_checkpoint)
    .trigger(availableNow=True)
    .start()
)


query_locations.awaitTermination()

## OpenAQ Sensors

In [0]:
bronze_openaq_measurements = spark.readStream.table("bg_traffic.bg_traffic_bronze.openaq_measurements")


In [0]:
measurements_checkpoint = "/Volumes/bg_traffic/bg_traffic_silver/checkpoints/openaq_measurements"

### Flattening

In [0]:
measurements_flatten = bronze_openaq_measurements\
    .select(
        F.col("fetched_at"),
        F.col("_ingestion_timestamp"),
        F.col("_source_file"),
        F.explode("data.results").alias("measurement")
    ).select(
        "fetched_at",
        "_ingestion_timestamp",
        "_source_file",
        F.col("measurement.locationsId").alias("location_id"),
        F.col("measurement.sensorsId").alias("sensor_id"),
        F.col("measurement.datetime.utc").alias("measurement_timestamp"),
        F.col("measurement.value").alias("measurement_value")
    )




### Casting

In [0]:
measurements_types = measurements_flatten\
    .withColumn("fetched_at", F.to_timestamp("fetched_at"))\
    .withColumn("measurement_timestamp", F.to_timestamp("measurement_timestamp"))\
    .withColumn("location_id", F.col("location_id").cast("long"))\
    .withColumn("sensor_id",F.col("sensor_id").cast("long"))\
    .withColumn("measurement_value", F.col("measurement_value").cast("double"))


measurements_types.printSchema()

### Validate

In [0]:
measurements_valid = measurements_types.filter(
    (F.col("location_id").isNotNull()) &
    (F.col("sensor_id").isNotNull()) &
    (F.col("measurement_timestamp").isNotNull()) &
    (F.col("measurement_value").isNotNull()) &
    (~F.isnan(F.col("measurement_value"))) 
)


### Merge & Dedup

In [0]:
def merge_measurements(df_source, batch_id):

    if df_source.isEmpty():
        return

    measurements_window = Window.partitionBy("location_id", "sensor_id", "measurement_timestamp")\
    .orderBy(F.col("fetched_at").desc())

    measurements_dedup = df_source\
    .withColumn("row_num", F.row_number().over(measurements_window))\
        .filter(F.col("row_num") == 1)\
            .drop("row_num")
            
    silver_table_measurements = DeltaTable.forName(
        spark, 
        "bg_traffic.bg_traffic_silver.openaq_measurements"
    ).alias("target").merge(measurements_dedup.alias("source"), 
                            """
                            target.location_id = source.location_id
                            AND target.sensor_id = source.sensor_id
                            AND target.measurement_timestamp = source.measurement_timestamp
                            """
                            )\
                                .whenMatchedUpdateAll()\
                                    .whenNotMatchedInsertAll()\
                                        .execute()


In [0]:
query_measurements = (
    measurements_valid
    .writeStream
    .foreachBatch(merge_measurements)
    .option("checkpointLocation", measurements_checkpoint)
    .trigger(availableNow=True)
    .start()
)


query_measurements.awaitTermination()

In [0]:
%sql
SELECT * FROM bg_traffic.bg_traffic_silver.openaq_locations;

In [0]:
%sql
SELECT * FROM bg_traffic.bg_traffic_silver.openaq_measurements;